# Neural Fortran Examples: Eleven Optimizers on a Quadratic

Created with Grok Build

Source code by [Milan Curcic](https://github.com/milancurcic) and the neural-fortran contributors, from the [neural-fortran](https://github.com/modern-fortran/neural-fortran) repository. The library is released under the MIT license. Copyright (c) 2018-2025 neural-fortran contributors. The paper is Milan Curcic, [A parallel Fortran framework for neural networks and deep learning](https://doi.org/10.1145/3323057.3323059), *ACM SIGPLAN Fortran Forum* 38(1), 2019.

## Introduction

This is `example/quadratic.f90`. The program name is `quadratic_fit`. It builds eleven identical copies of a one-input, three-hidden-unit, one-output network and fits the same quadratic with SGD, momentum, Nesterov, batch and minibatch gradient descent, RMSProp, three Adam settings, and two Adagrad settings.

## The function being fit

```{literalinclude} Fortran_Code/Section_Examples_Quadratic/app/quadratic.f90
:language: fortran
:lines: 111-115
```

The elemental function is

$$
y = \frac{1}{2}\left(\frac{x^2}{2} + \frac{x}{2} + 1\right) = \frac{x^2}{4} + \frac{x}{4} + \frac{1}{2}.
$$

At $x = 0$, $y = 1/2$. At $x = 1$, $y = 1$. For $x > 1$, $y > 1$. Every dense layer in this program uses the default sigmoid, so the network output lies in $(0, 1)$. On any test point past $x = 1$ the target is outside the range of the model. That ceiling shows up in the losses below.


## Data, copies, and the eleven calls

```{literalinclude} Fortran_Code/Section_Examples_Quadratic/app/quadratic.f90
:language: fortran
:lines: 12-107
```

There are 1000 training points. Each $x$ is uniform on $[0, 2)$, and $y$ is the quadratic. The test grid is the 100 points $x_i = (i-1) \cdot 2 / 100$, the same half-open interval, including values above 1.

`net(1)` is `input(1)`, `dense(3)`, `dense(1)`, which is $(1\cdot 3+3)+(3\cdot 1+1) = 10$ parameters. `net(2:) = net(1)` copies that initial draw to the other ten networks, so the optimizers start from the same weights. Training uses 1000 epochs and learning rate 0.01. Momentum, where it is used, is 0.9. RMSProp's decay is 0.9. Adam uses $\beta_1 = 0.85$, $\beta_2 = 0.95$, and $\epsilon = 10^{-8}$, rather than the type defaults 0.9, 0.999, and $10^{-8}$.


## The number that is printed

```{literalinclude} Fortran_Code/Section_Examples_Quadratic/app/quadratic.f90
:language: fortran
:lines: 164-168
```

Every optimizer prints the same line, and the label says RMSE. The expression under the label has no square root:

$$
\frac{1}{N}\sum_{i=1}^{N}(\hat y_i - y_i)^2.
$$

That is the mean squared error on the 100 test points. The root-mean-square error would be the square root of this number. The figure and the discussion use the value the program actually prints. A printed 0.142617 is an MSE. Its square root is about 0.378.


## SGD, with and without momentum

```{literalinclude} Fortran_Code/Section_Examples_Quadratic/app/quadratic.f90
:language: fortran
:lines: 117-174
```

SGD updates once per training sample. For each epoch and each $i$, the network takes `forward([x(i)])`, `backward([y(i)])`, and `update` with an `sgd` optimizer. With momentum left at 0, the library step is

$$
\theta \leftarrow \theta - \eta g.
$$

With momentum $\mu > 0$ the velocity and the parameter step in `minimize_sgd` are

$$
v \leftarrow \mu v - \eta g.
$$

Classical momentum then adds $v$ to $\theta$. Nesterov adds $\mu v - \eta g$ instead:

$$
\theta \leftarrow \theta + \mu v - \eta g.
$$

The three calls pass no momentum, momentum 0.9, and momentum 0.9 with `nesterov=.true.`. Each of those 1000 epochs performs 1000 updates. The printed heading is the same string, `Stochastic gradient descent`, for all three.


## Batch and minibatch

```{literalinclude} Fortran_Code/Section_Examples_Quadratic/app/quadratic.f90
:language: fortran
:lines: 176-269
```

Batch gradient descent accumulates `forward` and `backward` over all 1000 samples, then calls `update` once. The learning rate passed in is $\eta / N$, and `update` is called without a batch size, so the summed gradient is scaled by the caller rather than inside `update`.

The minibatch routine uses `batch_size = 100`, so an epoch has 10 updates. The learning rate passed to each update is $\eta / 100$. `batch_x` and `batch_y` are allocated and never read. The inner loop indexes `x` and `y` directly.

The batch order is a list of start positions, $1, 101, \ldots$, shuffled once with Fisher-Yates before the epoch loop. It is not reshuffled. The comment in the source says that `-O3` on gfortran needs `-fno-frontend-optimize` for this routine to converge. This release build still printed a decreasing minibatch curve, from 0.241941 at epoch 100 to 0.196503 at epoch 1000.


## RMSProp, Adam, and Adagrad

```{literalinclude} Fortran_Code/Section_Examples_Quadratic/app/quadratic.f90
:language: fortran
:lines: 271-432
```

These three routines also accumulate the gradient over the whole training set and call `update` once per epoch. RMSProp, with decay $\rho = 0.9$ and the type default $\epsilon = 10^{-8}$, updates a running mean of squared gradients and steps by

$$
r \leftarrow \rho r + (1-\rho)\, g^{2}, \qquad \theta \leftarrow \theta - \frac{\eta\, g}{\sqrt{r+\epsilon}}.
$$

Adam's step in `minimize_adam`, writing $g$ for the gradient plus the L2 term $\lambda_{\mathrm{L2}}\theta$, is

$$
m \leftarrow \beta_1 m + (1-\beta_1) g, \qquad v \leftarrow \beta_2 v + (1-\beta_2) g^{2},
$$

$$
\hat m = \frac{m}{1-\beta_1^{t}}, \qquad \hat v = \frac{v}{1-\beta_2^{t}}, \qquad \theta \leftarrow \theta - \eta\left(\frac{\hat m}{\sqrt{\hat v}+\epsilon} + \lambda_{\mathrm{dec}}\,\theta\right).
$$

`t` advances inside `minimize`, and `update` calls `minimize` once for the weights and once for the biases, so those two calls see consecutive values of `t`. The second Adam call sets $\lambda_{\mathrm{L2}} = 10^{-4}$. The third sets $\lambda_{\mathrm{dec}} = 10^{-5}$ and leaves the L2 term at 0. At these sizes the three Adam curves stay within $0.000001$ of each other at epoch 1000.

Adagrad accumulates $s \leftarrow s + g^{2}$ and steps by $\eta_t g / (\sqrt{s}+\epsilon)$, with

$$
\eta_t = \frac{\eta}{1+(t-1)\,\delta}.
$$

The last call sets $\lambda_{\mathrm{L2}} = 10^{-4}$ and $\delta = 0.99$. The decay is applied with the same fast-moving `t`, so the step size collapses. The plain Adagrad call leaves $\delta$ at 0.


## Shuffle

```{literalinclude} Fortran_Code/Section_Examples_Quadratic/app/quadratic.f90
:language: fortran
:lines: 434-447
```

`shuffle` is Fisher-Yates. `random_number` returns a value in $[0, 1)$, `floor(a * i) + 1` is an index from 1 through `i`, and the chosen entry is swapped with position `i`. Only the minibatch start positions go through it.

## Program

```{literalinclude} Fortran_Code/Section_Examples_Quadratic/app/quadratic.f90
:language: fortran
```

The package file below builds this source with fpm. The Fortran program itself is the file from the repository.

```{literalinclude} Fortran_Code/Section_Examples_Quadratic/fpm.toml
:language: toml
```

## Build

The program above was compiled from the neural-fortran repository at commit `357e63d` (`Support of the argument stride for locally_connected2d_layer`) with fpm 0.10.1:

```text
fpm build --profile release
```

The compiler was gfortran 14.2.0. The log is that release build. It compiles the library and every program in `example/`, including this one. The `fpm.toml` on this page is a standalone package for the copied source. It takes neural-fortran as a git dependency, in the same form as the sine-wave page.


```{literalinclude} Fortran_Code/fpm_release_build.log
:language: text
```

## Run

The binary was `quadratic` from the release build. Standard error was empty and the process exited 0. The run took about 6 seconds. The three SGD blocks and the three Adam blocks share a heading. They are identified by the call order in the source, which is the order of the curves in the figure.

```{literalinclude} Fortran_Code/Section_Examples_Quadratic/quadratic.out
:language: text
```

## What the run did

![Optimizer comparison on the quadratic](images/nf_quadratic.png)

Final printed mean squared error after 1000 epochs:

| Optimizer | Printed MSE |
| --- | ---: |
| SGD | 0.142617 |
| SGD, momentum 0.9 | 0.142511 |
| SGD, Nesterov | 0.142237 |
| batch gradient descent | 0.241946 |
| minibatch, size 100 | 0.196503 |
| RMSProp | 0.142141 |
| Adam | 0.146180 |
| Adam, L2 $10^{-4}$ | 0.146181 |
| Adam, decoupled decay $10^{-5}$ | 0.146180 |
| Adagrad | 0.253065 |
| Adagrad, L2 and learning-rate decay | 0.502408 |

RMSProp is the lowest of the eleven, and the three per-sample SGD variants sit next to it. Batch and minibatch descent make far fewer updates, because the learning rate is divided by 1000 or by 100 and the step happens once per epoch or once per 100 samples. Adagrad with weight decay and $\delta = 0.99$ stays near 0.50, which is the scale of the error before the fit has moved.

None of the sigmoid networks can drive this error to zero. Half of the test interval has $y > 1$, and a sigmoid output cannot land there. The cluster near 0.142 is that limit, reached by the optimizers that actually move, not a sign that those six runs found identical weights.


## Conclusion

One array of networks, copied from a single initializer, is enough to compare the optimizers in `nf_optimizers` on the same data. Read the printed column as mean squared error. The subroutine comment on minibatch convergence under `-O3` is worth keeping in mind, and this particular release run still descended.